# Olist — Pre-processamento

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import os

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

PASTA  = 'C:/Users/eymar/Downloads/olist/archive/'
df_raw = pd.read_csv(PASTA + 'dataset_features.csv')
df     = df_raw.copy()

print(f'{df.shape[0]:,} linhas x {df.shape[1]} colunas')
df.head(3)

## 1. Limpeza e Filtragem

In [ ]:
shape_antes = df.shape
print(f'Shape inicial: {shape_antes[0]:,} x {shape_antes[1]}')
print('-' * 50)

# 1a. Duplicatas
n_dup = df.duplicated().sum()
df = df.drop_duplicates()
print(f'Duplicatas removidas:              {n_dup}')

# 1b. Target invalido (dias <= 0)
n_inv = (df['dias_entrega'] <= 0).sum()
df = df[df['dias_entrega'] > 0]
print(f'Linhas com dias_entrega <= 0:      {n_inv}')

# 1c. Outliers extremos do target (acima do P99.5)
p995 = df['dias_entrega'].quantile(0.995)
n_ext = (df['dias_entrega'] > p995).sum()
df = df[df['dias_entrega'] <= p995]
print(f'Outliers do target (> {p995:.0f} dias):   {n_ext}')

# 1d. Valores faltantes
missing = df.isnull().sum()
missing = missing[missing > 0]
if missing.empty:
    print('Valores faltantes:                 0 — ok')
else:
    print(f'Valores faltantes encontrados:')
    print(missing.to_string())
    for col in missing.index:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            df[col] = df[col].fillna(df[col].median())
    print('Preenchidos com mediana (numericos) e moda (categoricos).')

print(f'\nShape apos limpeza: {df.shape[0]:,} x {df.shape[1]}')
print(f'Linhas removidas:   {shape_antes[0] - df.shape[0]:,}')

## 2. Remover Features Desnecessarias

In [ ]:
removidos = []

# 2a. Identificadores
ids = ['order_id', 'customer_id', 'seller_id']
for col in ids:
    if col in df.columns:
        df = df.drop(columns=col)
        removidos.append(f'{col} (identificador)')

# 2b. Features redundantes (multicolinearidade)
redundantes = ['valor_pago_total', 'seller_state']
for col in redundantes:
    if col in df.columns:
        df = df.drop(columns=col)
        removidos.append(f'{col} (redundante)')

print(f'Features removidas ({len(removidos)}):')
for c in removidos:
    print(f'  - {c}')

print(f'\nShape: {df.shape[0]:,} x {df.shape[1]}')
print(f'\nColunas restantes ({df.shape[1]}):')
for i, c in enumerate(df.columns, 1):
    print(f'  {i:2d}. {c}')

## 3. Tratar Outliers

In [ ]:
# Features que NAO serao clipadas (binarias, ordinais curtas, target)
nao_clipar = [
    'dias_entrega', 'eh_fim_de_semana', 'multiplos_vendedores', 'mesmo_estado',
    'tem_descricao', 'dia_semana_compra', 'mes_compra', 'hora_compra',
    'qtd_itens', 'qtd_vendedores', 'qtd_parcelas', 'qtd_tipos_pagamento'
]
colunas_clip = [
    c for c in df.select_dtypes(include='number').columns
    if c not in nao_clipar
]

# Snapshot antes do clip para visualizacao
df_pre_clip = df[colunas_clip].copy()

# Aplicar winsorization
limites_clip = {}
n_mod = {}
for col in colunas_clip:
    p01 = df[col].quantile(0.01)
    p99 = df[col].quantile(0.99)
    n_mod[col] = ((df[col] < p01) | (df[col] > p99)).sum()
    df[col] = df[col].clip(lower=p01, upper=p99)
    limites_clip[col] = (round(p01, 4), round(p99, 4))

# Visualizacao: boxplots antes x depois das 6 features mais impactadas
top6 = sorted(n_mod, key=n_mod.get, reverse=True)[:6]
fig, axes = plt.subplots(2, 6, figsize=(20, 6))
fig.suptitle('Outliers — Antes vs Depois do Clip (P01-P99)', fontweight='bold')

for i, col in enumerate(top6):
    kw_box  = dict(patch_artist=True, flierprops=dict(marker='.', markersize=2, alpha=0.3))
    axes[0, i].boxplot(df_pre_clip[col].dropna(), **kw_box,
                       boxprops=dict(facecolor='tomato', alpha=0.6))
    axes[0, i].set_title(col[:20], fontsize=8)
    axes[0, i].set_xticks([])
    if i == 0: axes[0, i].set_ylabel('Antes', fontsize=9)

    axes[1, i].boxplot(df[col].dropna(), **kw_box,
                       boxprops=dict(facecolor='steelblue', alpha=0.6))
    axes[1, i].set_xticks([])
    if i == 0: axes[1, i].set_ylabel('Depois', fontsize=9)

plt.tight_layout()
plt.show()

# Resumo
print(f'Winsorization aplicada em {len(colunas_clip)} features:')
print(f'{"Feature":<35s}  Corrigidos   P01         P99')
print('-' * 70)
for col in sorted(n_mod, key=n_mod.get, reverse=True):
    if n_mod[col] > 0:
        p01, p99 = limites_clip[col]
        print(f'{col:<35s}  {n_mod[col]:>8,}   {p01:>10.4f}  {p99:>10.4f}')

## 4. Transformacao Log

In [ ]:
# Features candidatas (excluir binarias, ordinais curtas, ciclicas)
nao_log = [
    'dias_entrega', 'eh_fim_de_semana', 'multiplos_vendedores', 'mesmo_estado',
    'tem_descricao', 'dia_semana_compra', 'mes_compra', 'hora_compra',
    'qtd_itens', 'qtd_vendedores', 'qtd_parcelas', 'qtd_tipos_pagamento', 'media_qtd_fotos'
]
candidatas = [
    c for c in df.select_dtypes(include='number').columns
    if c not in nao_log
]

# Identificar features com |skew| > 1 e min >= 0
features_log = []
for col in candidatas:
    sk = df[col].skew()
    if abs(sk) > 1 and df[col].min() >= 0:
        features_log.append((col, round(sk, 3)))

if not features_log:
    print('Nenhuma feature com |skewness| > 1. Nenhuma transformacao aplicada.')
else:
    cols_log = [c for c, _ in features_log]

    # Snapshot antes da transformacao
    snap_pre = df[cols_log].copy()

    # Aplicar log1p
    for col, _ in features_log:
        df[col] = np.log1p(df[col])

    # Visualizacao: histogramas antes x depois (top 6 mais assimetricos)
    n_show = min(6, len(features_log))
    top_features = sorted(features_log, key=lambda x: abs(x[1]), reverse=True)[:n_show]

    fig, axes = plt.subplots(2, n_show, figsize=(n_show * 3.5, 6), squeeze=False)
    fig.suptitle('Transformacao Log1p: Antes vs Depois', fontweight='bold')

    for i, (col, sk_a) in enumerate(top_features):
        sk_d = df[col].skew()
        axes[0, i].hist(snap_pre[col].dropna(), bins=40, color='tomato',
                        edgecolor='white', alpha=0.8)
        axes[0, i].set_title(f'{col[:18]}\nskew={sk_a:.2f}', fontsize=8)
        if i == 0: axes[0, i].set_ylabel('Antes', fontsize=9)

        axes[1, i].hist(df[col].dropna(), bins=40, color='steelblue',
                        edgecolor='white', alpha=0.8)
        axes[1, i].set_title(f'log1p\nskew={sk_d:.2f}', fontsize=8)
        if i == 0: axes[1, i].set_ylabel('Depois', fontsize=9)

    plt.tight_layout()
    plt.show()

    # Tabela resumo
    print(f'Features transformadas com log1p ({len(features_log)}):')
    print(f'{"Feature":<35s}  Skew Antes   Skew Depois')
    print('-' * 62)
    for col, sk_a in sorted(features_log, key=lambda x: abs(x[1]), reverse=True):
        sk_d = df[col].skew()
        print(f'{col:<35s}  {sk_a:>+10.2f}   {sk_d:>+10.2f}')

## 5. Encoding das Categoricas

In [ ]:
# 5a. One-Hot Encoding (baixa cardinalidade)
cat_ohe = [c for c in ['tipo_pagamento', 'regiao_cliente'] if c in df.columns]
df = pd.get_dummies(df, columns=cat_ohe, drop_first=False, dtype=int)

cols_ohe_novas = [c for c in df.columns
                  if any(c.startswith(p + '_') for p in cat_ohe)]
print(f'One-Hot Encoding em: {cat_ohe}')
print(f'  Colunas criadas ({len(cols_ohe_novas)}): {cols_ohe_novas}')

# 5b. Target Encoding (alta cardinalidade)
# Mapeamento: media de dias_entrega por categoria
cat_target = [c for c in ['categoria_produto', 'customer_state'] if c in df.columns]
target_maps = {}
media_global = df['dias_entrega'].mean()

for col in cat_target:
    media_por_cat = df.groupby(col)['dias_entrega'].mean()
    target_maps[col] = media_por_cat.to_dict()
    novo = col + '_enc'
    df[novo] = df[col].map(media_por_cat).fillna(media_global)
    df = df.drop(columns=col)
    print(f'Target Encoding: {col} -> {novo}  ({media_por_cat.shape[0]} categorias)')

print(f'\nShape apos encoding: {df.shape[0]:,} x {df.shape[1]}')
print(f'\nColunas atuais:')
for i, c in enumerate(df.columns, 1):
    print(f'  {i:2d}. {c}')

## 6. Normalizacao e Padronizacao

In [ ]:
# Colunas que NAO serao normalizadas
prefixos_ohe = [c for c in cat_ohe]   # tipo_pagamento, regiao_cliente
binarias = ['eh_fim_de_semana', 'multiplos_vendedores', 'mesmo_estado', 'tem_descricao']
ciclicas  = ['dia_semana_compra', 'mes_compra']

cols_nao_escalar = (
    [c for c in df.columns if any(c.startswith(p + '_') for p in prefixos_ohe)] +
    binarias + ciclicas + ['dias_entrega']
)

cols_escalar = [
    c for c in df.select_dtypes(include='number').columns
    if c not in cols_nao_escalar
]

print(f'Colunas que serao normalizadas ({len(cols_escalar)}):')
for c in sorted(cols_escalar):
    print(f'  - {c}')

print(f'\nColunas que NAO serao normalizadas:')
for c in sorted([cc for cc in df.columns if cc not in cols_escalar and cc != 'dias_entrega']):
    print(f'  - {c}')

print('\nO StandardScaler sera treinado no X_train e aplicado em X_train e X_test.')

## 7. Separar X e y — Train/Test Split

In [ ]:
# Separar X e y
X = df.drop('dias_entrega', axis=1)
y = df['dias_entrega']

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Tamanhos do split:')
print(f'  Total:   {len(df):,} amostras')
print(f'  Treino:  {len(X_train):,}  ({len(X_train)/len(df)*100:.0f}%)')
print(f'  Teste:   {len(X_test):,}   ({len(X_test)/len(df)*100:.0f}%)')
print(f'\nFeatures: {X.shape[1]}')
print(f'Target:   dias_entrega')
print(f'  Treino — media: {y_train.mean():.2f}  desvio: {y_train.std():.2f}')
print(f'  Teste  — media: {y_test.mean():.2f}  desvio: {y_test.std():.2f}')

# Normalizar: fit APENAS no treino, transform em ambos
X_train = X_train.copy()
X_test  = X_test.copy()

scaler = StandardScaler()
X_train[cols_escalar] = scaler.fit_transform(X_train[cols_escalar])
X_test[cols_escalar]  = scaler.transform(X_test[cols_escalar])

print('\nStandardScaler aplicado.')

# Salvar datasets e scaler
os.makedirs(PASTA, exist_ok=True)
X_train.to_csv(PASTA + 'X_train.csv', index=False)
X_test.to_csv( PASTA + 'X_test.csv',  index=False)
y_train.to_csv(PASTA + 'y_train.csv', index=False)
y_test.to_csv( PASTA + 'y_test.csv',  index=False)
joblib.dump(scaler,      PASTA + 'scaler.pkl')
joblib.dump(target_maps, PASTA + 'target_encoders.pkl')

print(f'\nArquivos salvos em {PASTA}:')
print(f'  X_train.csv         {X_train.shape[0]:,} x {X_train.shape[1]}')
print(f'  X_test.csv          {X_test.shape[0]:,}  x {X_test.shape[1]}')
print(f'  y_train.csv         {y_train.shape[0]:,}')
print(f'  y_test.csv          {y_test.shape[0]:,}')
print(f'  scaler.pkl')
print(f'  target_encoders.pkl')

## Resumo Final

In [ ]:
print('=' * 65)
print('  RESUMO DO PRE-PROCESSAMENTO')
print('=' * 65)
print(f'\nDataset original:     {df_raw.shape[0]:,} x {df_raw.shape[1]}')
print(f'Dataset final:        {df.shape[0]:,} x {df.shape[1]}')
print(f'  Linhas removidas:   {df_raw.shape[0] - df.shape[0]:,}')
print(f'  Features finais:    {X.shape[1]} features + 1 target')

print(f'\nPasso 1 — Limpeza')
print(f'  Filtragem do target (P99.5): {p995:.0f} dias')

print(f'\nPasso 2 — Features removidas: {len(removidos)}')
for c in removidos:
    print(f'  - {c}')

print(f'\nPasso 3 — Outliers (winsorization P01-P99): {len(colunas_clip)} features')

print(f'\nPasso 4 — Log1p aplicado: {len(features_log)} features')
for col, sk in features_log:
    print(f'  - {col}')

print(f'\nPasso 5 — Encoding')
print(f'  One-Hot: {cat_ohe}  ->  {len(cols_ohe_novas)} colunas')
print(f'  Target:  {cat_target}')

print(f'\nPasso 6 — StandardScaler: {len(cols_escalar)} features')

print(f'\nPasso 7 — Split 80/20')
print(f'  Treino: {len(X_train):,}  |  Teste: {len(X_test):,}')

print(f'\nFeatures prontas para o modelo ({X.shape[1]}):')
for i, c in enumerate(sorted(X.columns), 1):
    print(f'  {i:2d}. {c}')

print('\n' + '=' * 65)
print('  Proximo passo: modelo_olist.ipynb')
print('=' * 65)